In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers.advanced_activations import *
from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


From the best 21 configurations modes of respective categories are as follows.

**Initializer:** normal

**Layers:** 2

**Batch Size:** 64

**Optimizer:** adamax

**Shuffle:** True

**Scaler:** RobustScaler

**Loss:** mean_absolute_error

In [3]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [4]:
def radstimator(h1=20, h2=15, num_vars=5):
    init = 'normal'
    
    model = Sequential()
    model.add( Dense( h1, kernel_initializer=init, input_dim=num_vars ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( BatchNorm())
    model.add( Dense( h2, kernel_initializer=init ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( Dropout( rate=0.4 ))
    
    model.add( Dense( 1, kernel_initializer=init, activation='linear' ))
    
    return model

In [9]:
print(x_train6_[:5]) # 'Latitude', 'BSH', 'Temperature(avg)', 'Daylength', 'H0'

[[40.141       5.9         4.22916667  9.19499685 13.69287119]
 [40.141       1.3         7.6375      9.20626964 13.74607822]
 [40.141       0.          6.2375      9.21860396 13.80432176]
 [40.141       0.          3.32916667  9.23198817 13.86758193]
 [40.141       0.7         5.06956522  9.24640976 13.93583675]]


In [5]:
data6_3v_ = {'x_train': x_train6_[:, [1, 3, 4, 2]], 'x_dev': x_dev_[:, [1, 3, 4, 2]], 'x_test': x_test_[:, [1, 3, 4, 2]]}
data_3v_ = {'x_train': x_train_[:, [1, 3, 4, 2]], 'x_dev': x_dev_[:, [1, 3, 4, 2]], 'x_test': x_test_[:, [1, 3, 4, 2]]}

data6_3v = scale_data(RobustScaler(), data6_3v_)
data_3v = scale_data(RobustScaler(), data_3v_)

x_train6_3v = data6_3v['x_train']
x_train_3v = data_3v['x_train']

x_dev6_3v = data6_3v['x_dev']
x_dev_3v = data_3v['x_dev']

x_test6_3v = data6_3v['x_test']
x_test_3v = data_3v['x_test']

print(f'x_train shape: {x_train_3v.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_3v.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_3v.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_3v.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 4), y_train shape: (52416,)
x_train6 shape: (10654, 4), y_train6 shape: (10654,)
x_dev shape: (9011, 4), y_dev shape: (9011,)
x_test shape: (16899, 4), y_test shape: (16899,)


In [6]:

validation_data6 = ( x_dev6_3v, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 4)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-3vars-h/6stations-h-nNTH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train6_3v, y_train6, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train6_3v, batch_size = 64 )

    mse = MSE( y_train6, p )
    rmse = sqrt( mse )
    mae = MAE( y_train6, p )
    r2 = R2( y_train6, p )
    evs = EVS( y_train6, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 3.9668, MAE: 2.2465, R2: 0.8160, EVS: 0.8180, 
C02 » RMSE: 3.9459, MAE: 2.0996, R2: 0.8179, EVS: 0.8184, 
C03 » RMSE: 4.0068, MAE: 2.3126, R2: 0.8122, EVS: 0.8157, 
C04 » RMSE: 3.9807, MAE: 2.2350, R2: 0.8147, EVS: 0.8166, 
C05 » RMSE: 3.9561, MAE: 2.1313, R2: 0.8169, EVS: 0.8172, 
C06 » RMSE: 3.9764, MAE: 2.0294, R2: 0.8151, EVS: 0.8151, 
C07 » RMSE: 3.9704, MAE: 2.1961, R2: 0.8156, EVS: 0.8176, 
C08 » RMSE: 4.0824, MAE: 2.5810, R2: 0.8051, EVS: 0.8130, 
C09 » RMSE: 3.9643, MAE: 2.2191, R2: 0.8162, EVS: 0.8186, 
C10 » RMSE: 3.9664, MAE: 2.0139, R2: 0.8160, EVS: 0.8161, 
C11 » RMSE: 3.9535, MAE: 2.0482, R2: 0.8172, EVS: 0.8174, 
C12 » RMSE: 3.9586, MAE: 2.1686, R2: 0.8167, EVS: 0.8176, 
C13 » RMSE: 3.9592, MAE: 2.0312, R2: 0.8167, EVS: 0.8167, 
C14 » RMSE: 4.0227, MAE: 2.4625, R2: 0.8107, EVS: 0.8165, 
C15 » RMSE: 3.9313, MAE: 2.1281, R2: 0.8192, EVS: 0.8195, 
C16 » RMSE: 3.9456, MAE: 2.2010, R2: 0.8179, EVS: 0.8204, 
C17 » RMSE: 3.9754, MAE: 2.1019, R2: 0.8152, EVS: 0.8159

In [7]:
# Train with 6 stations data, and then with 31. Compare them.
# First for the variables n,N for H/H0.

validation_data = ( x_dev_3v, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 4)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-3vars-h/31stations-h-nNTH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train_3v, y_train, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train_3v, batch_size = 64 )

    mse = MSE( y_train, p )
    rmse = sqrt( mse )
    mae = MAE( y_train, p )
    r2 = R2( y_train, p )
    evs = EVS( y_train, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 3.0972, MAE: 1.8441, R2: 0.8712, EVS: 0.8740, 
C02 » RMSE: 3.0376, MAE: 1.7214, R2: 0.8761, EVS: 0.8771, 
C03 » RMSE: 3.0561, MAE: 1.7599, R2: 0.8746, EVS: 0.8771, 
C04 » RMSE: 3.1128, MAE: 1.9071, R2: 0.8699, EVS: 0.8750, 
C05 » RMSE: 3.0296, MAE: 1.7292, R2: 0.8768, EVS: 0.8772, 
C06 » RMSE: 3.0132, MAE: 1.6413, R2: 0.8781, EVS: 0.8781, 
C07 » RMSE: 3.0267, MAE: 1.6826, R2: 0.8770, EVS: 0.8776, 
C08 » RMSE: 3.0247, MAE: 1.6401, R2: 0.8772, EVS: 0.8774, 
C09 » RMSE: 3.0367, MAE: 1.7176, R2: 0.8762, EVS: 0.8778, 
C10 » RMSE: 3.0169, MAE: 1.7007, R2: 0.8778, EVS: 0.8779, 
C11 » RMSE: 3.0421, MAE: 1.7834, R2: 0.8757, EVS: 0.8781, 
C12 » RMSE: 3.0277, MAE: 1.7213, R2: 0.8769, EVS: 0.8775, 
C13 » RMSE: 3.0228, MAE: 1.6106, R2: 0.8773, EVS: 0.8773, 
C14 » RMSE: 3.0201, MAE: 1.6941, R2: 0.8775, EVS: 0.8782, 
C15 » RMSE: 3.0329, MAE: 1.7064, R2: 0.8765, EVS: 0.8781, 
C16 » RMSE: 3.0085, MAE: 1.6692, R2: 0.8785, EVS: 0.8788, 
C17 » RMSE: 3.0360, MAE: 1.6773, R2: 0.8762, EVS: 0.8767